# RotationnelMagnetique — `rotationnel_magnetique`

`Postraitement_fields` · ASN 28 validation form (TRUST-LiMe).



## Introduction

Validation of the **differential** post-treatment fields of the standalone
magnetic (induction) equation:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

This notebook runs **one and only one test case**: `rotationnel_magnetique`. Per the
TRUST validation-form layout the other `Postraitement_fields` cases live in
their own sibling folders.

## What it validates

$$
\mathbf{w} = \nabla \times \mathbf{b}, \qquad
w_z = \frac{\partial b_y}{\partial x} - \frac{\partial b_x}{\partial y}
$$
Exactly linear field $b = (-y, x)$: $\nabla b = \begin{pmatrix}0&-1\\1&0\end{pmatrix}$,
so the only nonzero component is the out-of-plane $w_z = 1 - (-1) = 2$. In 2D the
curl is a scalar (pseudoscalar) field.


## Contract checked

| # | Quantity | Formula |
|---|---|---|
| 1 | magnetic curl (vorticity) | $w = \nabla\times b $ |

All these fields are built from the module-reconstructed element gradient
$\nabla b$ (`gradient_b_elements`, Green-Gauss — the kernel has no
element-field gradient factory on PolyMAC). Vector model required: the
scalar model is rejected with a clean error by `creer_champ`.



In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()



## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.



In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case`."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def read_son_all(case, sonde):
    """Like read_son but returns every entry (components of a vector field)."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    return [(e, np.asarray(son.getValues(e)[1], dtype=float)) for e in son.getEntries()]

print("BUILD =", BUILD)



## Run

Sequential-only: the dataset is run by `trustutils` on one processor
(no `partition()`, no `nbProcs`).



In [ ]:
import pathlib, shutil

from trustutils import run

run.reset()
run.addCase(".", "rotationnel_magnetique.data")      # sequential-only: never nbProcs > 1
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob("rotationnel_magnetique_*.son"):
    shutil.copy2(son, build / son.name)



## Checks


In [ ]:
t, rot = read_son("rotationnel_magnetique", "SONDE_ROT")

# b = (-y, x) is exactly linear => w_z = d by/dx - d bx/dy = 1 - (-1) = 2.
expected = 2.0
print(f"rotationnel_magnetique (probe): {rot[-1]:.6g}  (expected {expected:.6g})")
err = abs(rot[-1] - expected) / expected
print(f"relative error                : {err:.3e}")
assert err < 1e-12, f"rotationnel_magnetique wrong: {rot[-1]}"
print("OK: rotationnel_magnetique = curl b (2D scalar).")


## Computer Performance


In [ ]:
run.tablePerf()

